# Testing: Synthetic Data

The `synthetic` module allows us to create artificial datasets with known velocities, extents etc. which we can then compare with those estimated by THUNER.

In [ ]:
"""Synthetic data demo/test."""

%load_ext autoreload
%autoreload 2
import xarray as xr
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
import numpy as np
import thuner.data as data
import thuner.default as default
import thuner.track.track as track
import thuner.option as option
import thuner.analyze as analyze
import thuner.data.synthetic as synthetic
import thuner.attribute as attribute
import thuner.visualize as visualize
from thuner.utils import format_time, copy_to_gallery
from thuner.log import setup_logger

logger = setup_logger(__name__)

## Geographic Coordinates

In [ ]:
# Parent directory for saving outputs
base_local = Path.home() / "THUNER_output"
start = "2005-11-13T00:00:00"
end = "2005-11-13T03:00:00"

# Set a flag for whether or not to remove existing output directories
remove_existing_outputs = True
output_parent = base_local / "runs/synthetic/geographic"

In [ ]:
if output_parent.exists() and remove_existing_outputs:
    shutil.rmtree(output_parent)

In [ ]:
options_directory = output_parent / "options"
options_directory.mkdir(parents=True, exist_ok=True)

# Create a grid
lat = np.arange(-14, -6 + 0.025, 0.025).tolist()
lon = np.arange(128, 136 + 0.025, 0.025).tolist()
grid_options = option.grid.GridOptions(name="geographic", latitude=lat, longitude=lon)
grid_options.to_json(options_directory / "grid.json")

# Initialize synthetic objects. Each is given a finite lifetime (30-120 min) and linear
# fade-in/out, so objects appear, intensify, weaken and disappear over the run.
starting_objects = []
for i in range(5):
    major = 3 * (7 + 4 * i)  # full axis length in km
    obj = synthetic.EllipsoidObject(
        time=start,
        center_latitude=np.mean(lat),
        center_longitude=lon[(i + 1) * len(lon) // 6],
        direction=-np.pi / 4 + i * np.pi / 8,
        speed=30 - 4 * i,
        major=major,
        minor=0.4 * major,
        orientation=0.25 * np.pi + i * np.pi / 8,
        life_time=120 + i * 30,
        fade_in_time=60,
        fade_out_time=60,
    )
    starting_objects.append(obj)
# Create data options dictionary. The objects are owned by a generator; FixedGenerator
# simply replays this fixed list (procedural generators are a future extension).
generator = synthetic.FixedGenerator(objects=starting_objects)
# target_objects tells analyze.synthetic.match_ground_truth which tracked object's
# masks to match the synthetic truth objects against (by centre containment).
synthetic_options = data.synthetic.SyntheticOptions(
    generator=generator, target_objects=["convective"]
)
data_options = option.data.DataOptions(datasets=[synthetic_options])
data_options.to_json(options_directory / "data.json")

track_options = default.track.synthetic_track()
track_options.to_json(options_directory / "track.json")

# Create the display_options dictionary
visualize_options = default.visualize.synthetic_runtime(
    options_directory / "visualize.json"
)
visualize_options.to_json(options_directory / "visualize.json")

In [ ]:
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    np.timedelta64(10, "m"),
)
track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=visualize_options,
    output_directory=output_parent,
)

In [ ]:
gif_filename = f"convective_{format_time(start, day_only=True)}.gif"
gif_filepath = output_parent / f"visualize/match/{gif_filename}"
copy_to_gallery(gif_filepath, gallery_name=f"synthetic_{gif_filename}")

![THUNER applied to synthetic data.](https://raw.githubusercontent.com/THUNER-project/THUNER/refs/heads/main/gallery/synthetic_convective_20051113.gif)

## Cartesian Coordinates

In [ ]:
central_latitude = -10
central_longitude = 132

y = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()
x = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()

grid_options = option.grid.GridOptions(
    name="cartesian",
    x=x,
    y=y,
    central_latitude=central_latitude,
    central_longitude=central_longitude,
)
grid_options.to_json(options_directory / "grid.json")

In [ ]:
output_parent = base_local / "runs/synthetic/cartesian"
if output_parent.exists() & remove_existing_outputs:
    shutil.rmtree(output_parent)
    
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    +np.timedelta64(10, "m"),
)

track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=None,
    output_directory=output_parent,
)

## Procedural scenes

Instead of placing objects by hand, a `RandomEllipseGenerator` spawns random cells over time: `initial_count` cells at the start, then new ones as a Poisson process at `spawn_rate` per hour, each with random geometry, motion and lifetime drawn from the configured ranges. It is deterministic given its `seed`, so re-running reproduces the same scene and the ground truth still matches the rendered data exactly. Procedural generators like this are the eventual aim of the synthetic module — building richer, more realistic scenes for testing.

In [ ]:
# A procedural scene over two hours, on the same geographic grid.
output_parent = base_local / "runs/synthetic/random"

In [ ]:
if output_parent.exists() and remove_existing_outputs:
    shutil.rmtree(output_parent)

In [ ]:
options_directory = output_parent / "options"
options_directory.mkdir(parents=True, exist_ok=True)

lat = np.arange(-14, -6 + 0.025, 0.025).tolist()
lon = np.arange(128, 136 + 0.025, 0.025).tolist()
grid_options = option.grid.GridOptions(name="geographic", latitude=lat, longitude=lon)
grid_options.to_json(options_directory / "grid.json")

generator = synthetic.RandomEllipseGenerator(
    seed=42,
    spawn_rate=8,  # ~8 new cells per hour
    initial_count=3,
    major_range=(30, 60),  # full major axis, km
    speed_range=(5, 25),  # m/s
    life_time_range=(30, 240),  # minutes
)
# target_objects tells analyze.synthetic.match_ground_truth which tracked object's
# masks to match the synthetic truth objects against (by centre containment).
synthetic_options = data.synthetic.SyntheticOptions(
    generator=generator,
    target_objects=["convective"],
    # Save each generated grid so attribute.series can reload it when plotting.
    converted_options={"save": True},
)
data_options = option.data.DataOptions(datasets=[synthetic_options])
data_options.to_json(options_directory / "data.json")

track_options = default.track.synthetic_track()
track_options.to_json(options_directory / "track.json")
visualize_options = default.visualize.synthetic_runtime(
    options_directory / "visualize.json"
)
visualize_options.to_json(options_directory / "visualize.json")

times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    np.timedelta64(10, "m"),
)
track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=None,
    output_directory=output_parent,
)

In [ ]:
analysis_options = analyze.mcs.AnalysisOptions()
analysis_options.to_json(options_directory / "analysis.json")
analyze.utils.smooth_flow_velocities("convective", output_parent)
analyze.utils.quality_control("convective", output_parent, analysis_options)

In [ ]:
style = "presentation"
attribute_handlers = default.visualize.detected_attribute_handlers(output_parent, style)
figure_options = option.visualize.HorizontalAttributeOptions(
    name='synthetic_convective',
    object_name='convective',
    style=style,
    attribute_handlers=attribute_handlers,
)
visualize.attribute.series(
    output_directory=output_parent,
    start_time=start,
    end_time=end,
    figure_options=figure_options,
    dataset_name="synthetic",
    parallel_figure=False,
    by_date=False,
    num_processes=8
)

In [ ]:
ground_truth = analyze.synthetic.write_ground_truth(
    output_parent, data_options=data_options, times=times, grid_options=grid_options
)
match_tables = analyze.synthetic.match_ground_truth(output_parent)

In [ ]:
core = attribute.utils.read_attribute(output_parent, "attributes", "convective", "core")
ellipse = attribute.utils.read_attribute(output_parent, "attributes", "convective", "ellipse")

In [ ]:
match_tables

In [ ]:
match_tables["synthetic"]

In [ ]:
core

In [ ]:
import matplotlib.pyplot as plt

# Compare each ground-truth object's known velocity with THUNER's flow-derived velocity
# for the object it was matched to. The match links a truth object (time, id) to a
# tracked object via convective_universal_id; u_flow/v_flow live in the convective
# "core" attributes, keyed by (time, universal_id), so we join on those.
truth = match_tables["synthetic"].reset_index()
truth = truth[truth["convective_universal_id"] != 0]  # keep only matched truth objects
flow = core.reset_index()[["time", "universal_id", "u_flow", "v_flow"]]
comparison = truth.merge(
    flow,
    left_on=["time", "convective_universal_id"],
    right_on=["time", "universal_id"],
    how="inner",
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, component in zip(axes, ["u", "v"]):
    true_velocity = comparison[component]
    flow_velocity = comparison[f"{component}_flow"]
    ax.scatter(true_velocity, flow_velocity, s=12, alpha=0.6)
    lims = [
        min(true_velocity.min(), flow_velocity.min()),
        max(true_velocity.max(), flow_velocity.max()),
    ]
    ax.plot(lims, lims, "k--", lw=1, label="1:1")
    ax.set_xlabel(f"true {component} [m/s]")
    ax.set_ylabel(f"matched {component}_flow [m/s]")
    ax.set_title(f"{component} velocity: truth vs THUNER")
    ax.set_aspect("equal")
    ax.legend()
fig.tight_layout()
plt.show()